# Task 2: Pop Melody Harmonization with Seq2Seq LSTM

## What is Harmonization?

Given a **melody**, harmonization generates an **accompaniment** that musically supports it.
Here we take the melody track from a pop song and generate a piano accompaniment.

## How This Differs from Task 1

| Aspect | Task 1 (Bach Chorales) | Task 2 (POP909) |
|--------|------------------------|-----------------|
| Genre | Baroque (1650–1750) | Modern pop |
| Dataset | ~370 chorales | 909 songs, 70k+ windows |
| Model | Decoder-only causal Transformer | Seq2Seq BiLSTM + Attention |
| Task | Unconditioned generation | Melody-conditioned generation |
| Melody access | Causal (no look-ahead) | Bidirectional (full melody visible to encoder) |
| Voices | 4 interleaved (SATB) | Melody → Piano (up to 4 voices) |

The key architectural insight: a harmonizer **should** see the full melody before
generating notes — something a causal decoder-only transformer cannot do.
A bidirectional encoder fixes this.


## 2. Exploratory Data Analysis

In [ ]:
import sys, os
sys.path.insert(0, 'modeling_task2')

import pretty_midi
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from collections import Counter

DATA_DIR = Path('data/POP909/POP909')
EDA_JSON = Path('EDA_task2/pop909_eda_results.json')

# Load pre-collected raw stats (computed over all 909 songs)
with open(EDA_JSON) as f:
    eda = json.load(f)

ds  = eda['dataset_info']
ms  = eda['melody_stats']
cs  = eda['chord_stats']
ss  = eda['song_stats']

print(f"Songs analyzed:   {ds['successful_analyses']} / {ds['total_songs_attempted']}")
print(f"Melody notes:     {ms['total_notes']:,}")
print(f"Pitch range:      MIDI {ms['pitch_min']} – {ms['pitch_max']}")
print(f"Mean pitch:       {ms['pitch_mean']:.1f}  (std {ms['pitch_std']:.1f})")
print(f"Unique chords:    {cs['unique_chord_types']}")
print(f"Chord instances:  {cs['total_chord_instances']:,}")
print(f"Avg song length:  {ss['length_mean']:.1f}s  (range {ss['length_min']:.0f}–{ss['length_max']:.0f}s)")
print(f"Avg note density: {ss['note_density_mean']:.2f} notes/s")


### 2.1 Parse a Single Song

Let's look at the raw MIDI structure for song 001.

In [ ]:
# Load one real song and inspect its tracks
song_dir = DATA_DIR / '001'
pm = pretty_midi.PrettyMIDI(str(song_dir / '001.mid'))

print(f"Instruments ({len(pm.instruments)} tracks):")
for i, inst in enumerate(pm.instruments):
    notes = inst.notes
    pitches = [n.pitch for n in notes]
    print(f"  Track {i}: program={inst.program}  notes={len(notes)}"
          f"  pitch_range=[{min(pitches)}, {max(pitches)}]"
          f"  duration={pm.get_end_time():.1f}s")

# Show first 5 chord annotations
print("\nFirst 5 chord annotations (chord_midi.txt):")
with open(song_dir / 'chord_midi.txt') as f:
    for line in list(f)[:5]:
        start, end, chord = line.strip().split('\t')
        print(f"  {float(start):.2f}s – {float(end):.2f}s  {chord}")


### 2.2 Melody Pitch Distribution

In [ ]:
all_pitches = eda['all_pitches']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(all_pitches, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(ms['pitch_mean'], color='red', linestyle='--', label=f"Mean: {ms['pitch_mean']:.1f}")
axes[0].set_xlabel('MIDI Pitch')
axes[0].set_ylabel('Note Count')
axes[0].set_title('Melody Pitch Distribution (all 909 songs)')
axes[0].legend()

# Pitch class distribution
pc_counts = Counter([p % 12 for p in all_pitches])
pc_names  = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
axes[1].bar(pc_names, [pc_counts[i] for i in range(12)], color='coral')
axes[1].set_xlabel('Pitch Class')
axes[1].set_ylabel('Count')
axes[1].set_title('Pitch Class Distribution')

plt.tight_layout()
plt.savefig('images/task2_pitch_dist_inline.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Total notes: {len(all_pitches):,}  |  Range: {min(all_pitches)}–{max(all_pitches)}")


### 2.3 Chord Vocabulary

POP909 has 176 unique chord types — far richer than Bach's functional harmony.

In [ ]:
top_chords = cs['top_20_chords']  # dict {chord: count}
chords = list(top_chords.keys())[:20]
counts = [top_chords[c] for c in chords]
total_instances = cs['total_chord_instances']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Bar chart top 20
axes[0].barh(chords[::-1], counts[::-1], color='mediumpurple')
axes[0].set_xlabel('Occurrences')
axes[0].set_title(f'Top 20 Chord Types (vocab size: {cs["unique_chord_types"]})')

# Chord quality breakdown
quality_counts = Counter()
for chord in eda['all_chord_types']:
    if ':' in chord:
        quality_counts[chord.split(':')[1]] += top_chords.get(chord, 0)
    else:
        quality_counts['N'] += top_chords.get(chord, 0)
top_qualities = quality_counts.most_common(10)
axes[1].bar([q for q,_ in top_qualities], [c for _,c in top_qualities], color='teal')
axes[1].set_xlabel('Chord Quality')
axes[1].set_ylabel('Total Occurrences')
axes[1].set_title('Occurrences by Chord Quality (top 10)')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('images/task2_chord_vocab_inline.png', dpi=150, bbox_inches='tight')
plt.show()

top5 = list(top_chords.items())[:5]
print("Top 5 chords:")
for chord, cnt in top5:
    pct = 100 * cnt / total_instances
    print(f"  {chord:<20} {cnt:>6,}  ({pct:.1f}%)")


### 2.4 Song Length & Note Density

In [ ]:
song_lengths    = eda['song_lengths']     # seconds
note_densities  = eda['note_densities']   # notes/sec

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(song_lengths, bins=40, color='goldenrod', edgecolor='white', linewidth=0.3)
axes[0].axvline(ss['length_mean'], color='red', linestyle='--', label=f"Mean: {ss['length_mean']:.0f}s")
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Number of Songs')
axes[0].set_title('Song Length Distribution')
axes[0].legend()

axes[1].hist(note_densities, bins=40, color='seagreen', edgecolor='white', linewidth=0.3)
axes[1].axvline(ss['note_density_mean'], color='red', linestyle='--',
                label=f"Mean: {ss['note_density_mean']:.2f} n/s")
axes[1].set_xlabel('Notes per Second')
axes[1].set_ylabel('Number of Songs')
axes[1].set_title('Melody Note Density Distribution')
axes[1].legend()

plt.tight_layout()
plt.savefig('images/task2_song_stats_inline.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. Data Pipeline

Each song is segmented into **64 16th-note windows** (≈ 4 bars at 120 BPM) with 50% overlap.

- **Melody:** single MIDI pitch per 16th-note slot, 128 = REST
- **Piano:** up to 4 simultaneous pitches per slot (sorted descending, 0 = pad)
- Windows with >80% REST in the melody are discarded


In [ ]:
from pop909_dataset import POP909Dataset
import torch

train_ds = POP909Dataset('data/POP909/POP909', split='train')
val_ds   = POP909Dataset('data/POP909/POP909', split='val')

print(f"Training windows:   {len(train_ds):,}")
print(f"Validation windows: {len(val_ds):,}")
print(f"Total windows:      {len(train_ds)+len(val_ds):,}")

melody, piano = train_ds[0]
print(f"\nSample shapes — melody: {tuple(melody.shape)}, piano: {tuple(piano.shape)}")
print(f"Melody tokens (first 16): {melody[:16].tolist()}")
print(f"Piano tokens  (first 4):  {piano[:4].tolist()}")

# Show how many windows come from REST filtering
unique_pitches = torch.unique(melody[melody != 128])
print(f"Unique non-REST pitches in this window: {len(unique_pitches)}")


## 4. Model: Seq2Seq BiLSTM with Bahdanau Attention

```
Melody tokens (64,)
      │
 [Embedding 130→128]
      │
 [BiLSTM ×2 layers, hidden=256]   ← bidirectional: sees full melody
      │
 encoder_outputs (64, 512)
      │
      ├──────────────────────────────────────┐
      │          BAHDANAU ATTENTION          │
      │  at each decode step t:              │
      │  score(h_t, encoder_j) = v·tanh(    │
      │    W1·h_t + W2·encoder_j)            │
      │  α = softmax(scores)                 │
      │  context = Σ α_j · encoder_j         │
      └──────────────────────────────────────┘
                        │
              [LSTM Decoder ×2 layers]
                        │
             ┌──────────┼──────────┐
          [Lin]      [Lin]      [Lin]
          voice0     voice1     voice2     (+ voice3)
          (0–128)   (0–128)   (0–128)
```

The bidirectional encoder is the key difference from Task 1's causal transformer:
it can use **future melody context** when deciding what harmony fits at each beat.


In [ ]:
from harmonizer_model import HarmonizerSeq2Seq
import torch

model = HarmonizerSeq2Seq(
    melody_vocab=130,
    piano_vocab=129,
    melody_embed_dim=128,
    melody_hidden_dim=256,
    chord_embed_dim=128,
    chord_hidden_dim=256,
    num_layers=2,
    dropout=0.3
)

total      = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
enc_params = sum(p.numel() for p in model.melody_encoder.parameters())
dec_params = sum(p.numel() for p in model.chord_decoder.parameters())

print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
print(f"  Encoder:            {enc_params:,}")
print(f"  Decoder:            {dec_params:,}")

# Quick forward pass sanity check
model.eval()
with torch.no_grad():
    dummy_melody = torch.randint(0, 128, (2, 64))
    dummy_piano  = torch.zeros(2, 64, 4, dtype=torch.long)
    out = model(dummy_melody, dummy_piano, teacher_forcing_ratio=0.0)
    print(f"\nForward pass OK — output shape: {out.shape}")
    print(f"  Expected: (batch=2, seq=64, voices=4, vocab=129)")


In [ ]:
# Training loss curve (available after Colab training)
import os, json, matplotlib.pyplot as plt

losses_path = 'modeling_task2/harmonizer_losses.json'
if os.path.exists(losses_path):
    with open(losses_path) as f:
        L = json.load(f)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(L['train_loss'], label='Train loss')
    ax.plot(L['val_loss'],   label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Harmonizer Training Curves')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('images/task2_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Best val loss: {min(L['val_loss']):.4f} at epoch {L['val_loss'].index(min(L['val_loss']))+1}")
else:
    print("No training checkpoint yet — run Colab training first.")
    print("  1. Upload modeling_task2/pop909_cache.pkl to Colab")
    print("  2. Run colab/task2/train_harmonizer_colab.py")
    print("  3. Download harmonizer_best.pt → modeling_task2/")


## 5. Colab Training

Training requires a GPU (~20-30 min on T4, ~10 min on A100).

### Steps
1. Open a new Colab notebook
2. Upload `modeling_task2/pop909_cache.pkl` (91 MB — pre-processed windows)
3. Upload `modeling_task2/harmonizer_model.py`, `modeling_task2/pop909_dataset.py`
4. Copy cells from `colab/task2/train_harmonizer_colab.py` into Colab
5. Run all cells
6. Download `harmonizer_best.pt` → save to `modeling_task2/`

Expected time: **T4 ≈ 30 min**, **A100 ≈ 10 min** for 30 epochs


## 6. Generated Samples

In [ ]:
import subprocess, base64
from pathlib import Path
from IPython.display import HTML, display

SOUNDFONT  = 'modeling/checkpoints/MuseScore_General.sf3'
EVAL_DIR   = Path('evaluation_task2')
CHECKPOINT = Path('modeling_task2/harmonizer_best.pt')

def midi_to_wav(midi_path, wav_path):
    r = subprocess.run(
        ['fluidsynth', '-ni', SOUNDFONT, str(midi_path), '-F', str(wav_path), '-r', '44100'],
        capture_output=True, timeout=60)
    return r.returncode == 0

def audio_html(wav_path, title):
    data = base64.b64encode(open(wav_path,'rb').read()).decode()
    return f'<p><b>{title}</b></p><audio controls><source src="data:audio/wav;base64,{data}" type="audio/wav"></audio>'

if not CHECKPOINT.exists():
    print("No checkpoint found — train on Colab first, then place harmonizer_best.pt in modeling_task2/")
else:
    # Generate for a few test songs
    for song_id in ['001', '042', '100']:
        midi_out = EVAL_DIR / f'harmony_{song_id}.mid'
        if not midi_out.exists():
            subprocess.run([
                sys.executable, 'modeling_task2/generate_harmony.py', '--song', song_id
            ], capture_output=True)

    html_parts = []
    for song_id in ['001', '042', '100']:
        for suffix, label in [('', 'Generated'), ('_original', 'Original')]:
            mid = EVAL_DIR / f'harmony_{song_id}{suffix}.mid'
            wav = mid.with_suffix('.wav')
            if mid.exists():
                if not wav.exists():
                    midi_to_wav(mid, wav)
                if wav.exists():
                    html_parts.append(audio_html(wav, f'Song {song_id} — {label}'))

    if html_parts:
        display(HTML('<br>'.join(html_parts)))
    else:
        print("MIDI files generated but WAV conversion failed (fluidsynth not installed?).")
        print("MIDI files are in evaluation_task2/ — open them in any MIDI player.")


## 7. Evaluation Metrics

We compare our model against a **random baseline** (randomly sampled pitches within the
detected key) using four objective metrics:

| Metric | Definition | Direction |
|--------|-----------|-----------|
| Scale Consistency | % notes fitting detected key | ↑ higher better |
| Voice Crossing Rate | % timesteps with ordering violations | ↓ lower better |
| Parallel 5ths Rate | % voice pairs with parallel perfect 5ths | ↓ lower better |
| Pitch KL Divergence | KL vs. real pop pitch distribution | ↓ lower better |


In [ ]:
import json, os
import pandas as pd

metrics_path = 'evaluation_task2/harmony_metrics.json'

if os.path.exists(metrics_path):
    with open(metrics_path) as f:
        metrics = json.load(f)

    rows = {}
    for model_name, vals in metrics.items():
        rows[model_name] = vals

    df = pd.DataFrame(rows).T
    print(df.round(3).to_string())
else:
    print("No metrics file yet — run after training:")
    print("  python evaluation_task2/evaluate_harmony.py")
    print()
    # Show placeholder table so the structure is clear
    placeholder = pd.DataFrame({
        'Our Model':        {'Scale Consistency (%)': '—', 'Voice Crossing (%)': '—',
                             'Parallel 5ths (%)': '—',   'Pitch KL':           '—'},
        'Random Baseline':  {'Scale Consistency (%)': 69.9, 'Voice Crossing (%)': 96.3,
                             'Parallel 5ths (%)': 0.0,   'Pitch KL':           0.10},
        'Real Pop (POP909)':{'Scale Consistency (%)': '~90+', 'Voice Crossing (%)': '~1',
                             'Parallel 5ths (%)': '~0.2', 'Pitch KL':           0.0},
    }).T
    print(placeholder.to_string())


## 8. Related Work

### DeepBach — Hadjeres et al. (2017)
Bach harmonization via **Gibbs sampling** over per-voice neural networks.
Each voice has its own model; inference iterates Gibbs sweeps until convergence.
- Pros: principled constraint enforcement, interpretable per-voice control
- Cons: slow (many Gibbs iterations), designed for Bach's strict rule system
- vs. ours: our single-pass seq2seq is ~100× faster at inference; less principled
  about voice-leading constraints but generalizes to pop's looser harmonic grammar

### Coconet — Huang et al. (2017)
Polyphonic music generation with **dilated CNNs + blocked Gibbs sampling**.
Treats all voices as an image and inpaints missing ones.
- Pros: parallel multi-voice generation, flexible masking strategy
- Cons: still requires iterative sampling; CNN limits long-range dependency
- vs. ours: attention gives us explicit long-range melody–harmony alignment

### Music Transformer — Huang et al. (2018)
Relative attention for long-form piano generation (Maestro dataset).
- Tackles the same *length* problem (long MIDI sequences) with relative positional embeddings
- Unconditioned — no melody conditioning
- This is closest to Task 1 (unconditioned Bach generation); Task 2 adds explicit conditioning

### Pop Music Harmonization
POP909-based models (e.g., Pop Music Transformer, 2020) use Transformer-XL for
long-form generation but mostly unconditioned. Our melody-conditioned seq2seq fills
a specific gap: given a fixed melody, generate complementary harmonic material —
closer in spirit to DeepBach but for pop.
